In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pickle
import time

theta_space = np.linspace(-1, 1, 10)
theta_dot_space = np.linspace(-10, 10, 10)

def get_state(observation):
    cos_theta1, sin_theta1, cos_theta2, sin_theta2, theta1_dot, theta2_dot = \
        observation
    c_th1 = int(np.digitize(cos_theta1, theta_space))
    s_th1 = int(np.digitize(sin_theta1, theta_space))
    c_th2 = int(np.digitize(cos_theta2, theta_space))
    s_th2 = int(np.digitize(sin_theta2, theta_space))
    th1_dot = int(np.digitize(theta1_dot, theta_dot_space))
    th2_dot = int(np.digitize(theta2_dot, theta_dot_space))

    return (c_th1, s_th1, c_th2, s_th2, th1_dot, th2_dot)

def maxAction(Q, state, actions=[0, 1, 2]):
    values = []
    for a in actions:
        tup = Q[state,a]
        if tup[1] ==0:
            values.append(tup[0])
        else:
            values.append(tup[0]/tup[1])
    action = values.index(max(values))
    return action

def e_greedy_policy(state,eps,Q):
    if np.random.random()<eps:
        action =env.action_space.sample()
    else:
        action = maxAction(Q,state)
    return action

env = gym.make('Acrobot-v1')
n_games = 50000
alpha = 0.01
gamma = 0.99
eps = 1

action_space = [0, 1, 2]

states = []
for c1 in range(11):
    for s1 in range(11):
        for c2 in range(11):
            for s2 in range(11):
                for dot1 in range(11):
                    for dot2 in range(11):
                        states.append((c1, s1, c2, s2, dot1, dot2))

Q = {}
for state in states:
    for action in action_space:
        Q[state, action] = (np.random.random(),0)
"""
pickle_in = open('acrobot.pkl', 'rb')
Q = pickle.load(pickle_in)
env = wrappers.Monitor(env, "tmp/acrobot", video_callable=lambda episode_id: True, force=True)
"""
eps_rewards = 0
total_rewards = np.zeros(n_games)
rewards_sum =0
all_rewards = []
global state_list, action_list, reward_list
start_time = time.time()
for i in range(n_games):
    
    observation , _ = env.reset()
    state = get_state(observation)
    done = False
    action = e_greedy_policy(state,eps,Q)

    eps_rewards = 0
    
    memory = []
    reward_list = []

    count = 0
     
    done = False
    while not done:

        action = e_greedy_policy(state,eps,Q)

        observation_, reward, done, _, _ = env.step(action)
        memory.append((state,action))        #write to memory
        reward_list.append(reward)
        eps_rewards+=reward
        state = get_state(observation_)
        count+=1
        
    G = 0
    update_count = 0
    for t, (state,action) in enumerate(reversed(memory[:-1])):
        G = gamma*G + reward_list[t]
        if (state,action) not in memory[:t-1]:
            tup = Q[state,action]
            Q[state,action] = (tup[0]+G,tup[1]+1)
            update_count +=1

    total_rewards[i] = eps_rewards  
    all_rewards.append(eps_rewards)
    episode_rewards_array = np.array(all_rewards)
    mean_last_50_episodes = np.mean(episode_rewards_array[-50:])
    if i % 10 == 0:
        print('episode ', i, 'score ', eps_rewards, 'eps', eps,'updates', update_count, 'mean_last_50_episodes', mean_last_50_episodes)
    #if update_count<50: print(state_list[0])
    eps = eps - 2 / n_games if eps > 0.01 else 0.01
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time} seconds")
mean_rewards = np.zeros(n_games)

for t in range(n_games):
    mean_rewards[t] = np.mean(total_rewards[max(0, t-50):(t+1)])
    
arrname = "MC_mean_rewards_training," + str(n_games) + "episodes.npy"
np.save(arrname,mean_rewards)
    
plt.plot(mean_rewards)
#plt.ylim(bottom=-400)
plt.xlabel('Episode')
plt.ylabel('Mean Reward')
plt.title("Monte carlo training, mean rewards over last 50 episodes")
figname = "MonteCarloTraining," + str(n_games) + "episodes.png"
#plt.savefig(figname)    #save training figure
plt.show()
env.close()

filename = "acrobot_MC_" + str(n_games) + ".pkl"

#f = open(filename,"wb") ##save Q matrix
#pickle.dump(Q,f)
#f.close()

In [ ]:
ep_rewards = []
for episode in range(500):
    observation , _ = env.reset()
    state = get_state(observation)
    done = False
    ep_reward = 0
    while not done:
        action = e_greedy_policy(state,eps,Q)
        observation, reward, done, _ , _= env.step(action)
        state = get_state(observation)
        ep_reward += reward
        #env.render()
    ep_rewards.append(ep_reward)
    #print(f"episode: {episode}, rewards: {ep_reward}")

mean = sum(ep_rewards) / len(ep_rewards)
print(mean)
#plt.plot(ep_rewards)
x = range(len(ep_rewards))
plt.scatter(x,ep_rewards, marker = 'o')
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.title('MC evaluation, 500 episode rewards')
plt.savefig('MC_3000_evaluate')
plt.show()
env.close()